# Y-gene omics comparison

Recalculate the plotted ratios from the selected supplied inputs, preserving study-specific normalization and eligibility. The first column uses the source-notebook fluorescence analysis reproduced in Dataset S7. Missing values remain missing; ratios are descriptive comparisons.

In [ ]:
from pathlib import Path
import numpy as np
import numpy.typing
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PACKAGE = Path.cwd()
if not (PACKAGE / "Datasets").is_dir():
    PACKAGE = PACKAGE.parent
DATASETS = PACKAGE / "Datasets"
if not DATASETS.is_dir():
    raise FileNotFoundError("Launch this notebook from the package root or Github folder.")
plt.rcParams.update({"font.family": "Arial", "text.usetex": False})

import csv, hashlib, io, json, math, platform
import matplotlib
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Rectangle
import openpyxl
DATA_FILE = DATASETS / "Dataset S8.xlsx"
CHECK_REFERENCE_MATRIX = True
FIGURE_WIDTH_MM = 180
FIGURE_HEIGHT_MM = 132

In [ ]:
workbook = openpyxl.load_workbook(DATA_FILE, data_only=True, read_only=True)
def table(name):
    iterator = workbook[name].iter_rows(values_only=True)
    headers = next(iterator)
    return [dict(zip(headers, row)) for row in iterator if any(v is not None for v in row)]

expected_sheets = {"Heatmap data", "Heatmap columns", "Measurements", "Source samples", "Reviewed sources"}
assert expected_sheets.issubset(workbook.sheetnames)
measurements = table("Measurements")
column_meta = sorted(table("Heatmap columns"), key=lambda row: int(row["column_order"]))
reference_rows = table("Heatmap data")
sources = table("Reviewed sources")
source_samples = table("Source samples")
assert len(column_meta) == 10 and len(reference_rows) == 19 and len(measurements) == 190
assert len(sources) == 8
assert len({row["assay_id"] for row in column_meta}) == 10
assert {row["study_id"] for row in sources} == {row["study_id"] for row in column_meta}
manzanarez_inputs = {}
for row in source_samples:
    if row["study_label"] == "Manzanarez 2026" and row["phase"] != "Sample metadata":
        series = row["sample_or_metric"].split(" | ")[0]
        key = (row["gene"], series, row["phase"])
        assert key not in manzanarez_inputs
        manzanarez_inputs[key] = row["used_value"]
assert len(manzanarez_inputs) == 19 * 2 * 3
print("Source sheets:", ", ".join(workbook.sheetnames))
print("Measurements:", len(measurements), "| Readouts:", len(column_meta), "| Study entries:", len(sources))

## Recalculate ratios
Phase means are arithmetic means on the linear scale. Manzanarez uses two calibrated acquisition series, not an assertion of two independent biological replicates.

In [ ]:
def positive_number(value):
    return isinstance(value, (int, float)) and math.isfinite(value) and value > 0

def phase_inputs(row):
    if row["study_id"] != "MANZANAREZ2026":
        return row["exponential_input"], row["stationary_input"]
    phase = "T1 (6 h)" if row["assay_id"].endswith("_T1") else "T2 (24 h)"
    exp = [manzanarez_inputs[(row["gene"], series, "Exponential reference")]
           for series in ("Group 1", "Group 2")]
    stat = [manzanarez_inputs[(row["gene"], series, phase)]
            for series in ("Group 1", "Group 2")]
    exp_mean = sum(exp) / 2 if all(positive_number(v) for v in exp) else None
    stat_mean = sum(stat) / 2 if all(positive_number(v) for v in stat) else None
    if CHECK_REFERENCE_MATRIX:
        for recalculated, cached in [(exp_mean, row["exponential_input"]), (stat_mean, row["stationary_input"])]:
            if recalculated is None:
                assert cached in (None, ""), (row["gene"], row["assay_id"])
            else:
                assert isinstance(cached, (int, float)) and math.isclose(recalculated, cached, abs_tol=1e-12)
    return exp_mean, stat_mean

def calculate_ratio(row):
    exp, stat = phase_inputs(row)
    valid = positive_number(exp) and positive_number(stat)
    if row["assay_id"] == "MORIN_HALFLIFE":
        cvs = [row["cv_exponential"], row["cv_stationary"]]
        valid = valid and all(isinstance(cv, (int, float)) and math.isfinite(cv)
                              and 0 <= cv <= 40 for cv in cvs)
    return math.log2(stat / exp) if valid else np.nan

ratios = {}
for row in measurements:
    key = (row["gene"], row["assay_id"])
    assert key not in ratios, f"Duplicate measurement {key}"
    ratios[key] = calculate_ratio(row)
    if CHECK_REFERENCE_MATRIX and row["provided_log2fc"] is not None:
        assert math.isfinite(ratios[key]) and math.isclose(ratios[key], row["provided_log2fc"], abs_tol=1e-10), key
for assay in ("MANZANAREZ_PROTEIN_T1", "MANZANAREZ_PROTEIN_T2"):
    assert sum(math.isfinite(value) for (gene, a), value in ratios.items() if a == assay) == 9
print("Recalculated", sum(math.isfinite(v) for v in ratios.values()), "finite ratios; missingness retained.")

In [ ]:
allowed_genes = {row["gene"] for row in reference_rows}
SORT_ASSAY = "THIS_STUDY_INTENSITY"
genes = sorted(allowed_genes, key=lambda gene: (ratios[(gene, SORT_ASSAY)], gene))
assert all(math.isfinite(ratios[(gene, SORT_ASSAY)]) for gene in genes)
assert not any(gene.lower() in {"control", "wt", "wild type", "wild-type"} for gene in genes)
assays = [row["assay_id"] for row in column_meta]
values = np.array([[ratios[(gene, assay)] for assay in assays] for gene in genes], dtype=float)
assert values.shape == (19, 10)
if CHECK_REFERENCE_MATRIX:
    reference = {row["gene"]: row for row in reference_rows}
    for i, gene in enumerate(genes):
        for j, assay in enumerate(assays):
            saved = reference[gene][assay]
            if saved in (None, ""):
                assert np.isnan(values[i, j]), (gene, assay)
            else:
                assert math.isclose(values[i, j], saved, abs_tol=1e-10), (gene, assay)
print("Gene order:", ", ".join(genes))
print("Finite plotted ratios:", int(np.isfinite(values).sum()), "| Missing/unreliable:", int(np.isnan(values).sum()))

## Display the heatmap

In [ ]:
groups = []
for (j, row) in enumerate(column_meta):
    if not groups or groups[-1]['id'] != row['group_id']:
        groups.append({'id': row['group_id'], 'start': j, 'end': j, 'label': row['group_label']})
    else:
        groups[-1]['end'] = j
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 7, 'axes.labelsize': 7, 'xtick.labelsize': 6.5, 'ytick.labelsize': 7.5, 'text.usetex': False, 'axes.linewidth': 0.4})
cmap = plt.get_cmap('RdBu_r').copy()
cmap.set_bad('#e5e5e5')
norm = TwoSlopeNorm(vmin=-6, vcenter=0, vmax=6)
assert cmap(norm(-3))[2] > cmap(norm(-3))[0]
assert cmap(norm(3))[0] > cmap(norm(3))[2]
fig = plt.figure(figsize=(FIGURE_WIDTH_MM / 25.4, FIGURE_HEIGHT_MM / 25.4))

def rectangle_mm(left, bottom, width, height):
    return [left / FIGURE_WIDTH_MM, bottom / FIGURE_HEIGHT_MM, width / FIGURE_WIDTH_MM, height / FIGURE_HEIGHT_MM]
(matrix_left_mm, matrix_width_mm) = (19, 144)
(matrix_bottom_mm, matrix_height_mm) = (35, 81)
panel_gap_mm = 6
cell_width_mm = (matrix_width_mm - panel_gap_mm * (len(groups) - 1)) / len(assays)
panel_left_mm = matrix_left_mm
(panels, annotations) = ([], [])
row_centres = np.arange(len(genes)) + 0.5

def figure_text_mm(x, y, label, **kwargs):
    return fig.text(x / FIGURE_WIDTH_MM, y / FIGURE_HEIGHT_MM, label, **kwargs)
for (group_index, group) in enumerate(groups):
    (start, stop) = (group['start'], group['end'] + 1)
    ncols = stop - start
    panel_width_mm = ncols * cell_width_mm
    ax = fig.add_axes(rectangle_mm(panel_left_mm, matrix_bottom_mm, panel_width_mm, matrix_height_mm))
    block = np.ma.masked_invalid(values[:, start:stop])
    mesh = ax.pcolormesh(np.arange(ncols + 1), np.arange(len(genes) + 1), block, shading='flat', cmap=cmap, norm=norm, edgecolors='white', linewidth=0.9, antialiased=False)
    ax.set_xlim(0, ncols)
    ax.set_ylim(len(genes), 0)
    ax.set_xticks([])
    ax.set_yticks(row_centres)
    if group_index == 0:
        ax.set_yticklabels(genes, fontstyle='italic')
        for label in ax.get_yticklabels():
            label.set_verticalalignment('center')
    else:
        ax.tick_params(labelleft=False)
    ax.tick_params(length=0, pad=4)
    for spine in ax.spines.values():
        spine.set_visible(False)
    heading = {'THIS_STUDY': 'Mean' + chr(10) + 'intensity', 'RNA': 'RNA abundance', 'PROTEIN': 'Protein abundance', 'HALFLIFE': 'mRNA' + chr(10) + 'half-life'}[group['id']]
    ax.text(0.5, 1.045, heading, transform=ax.transAxes, ha='center', va='bottom', fontsize=7.5, linespacing=1.2, clip_on=False)
    ax.plot([0.06, 0.94], [1.035, 1.035], transform=ax.transAxes, color='#333333', linewidth=0.6, clip_on=False)
    j = start
    while j < stop:
        end = j + 1
        while end < stop and column_meta[end]['study_id'] == column_meta[j]['study_id']:
            end += 1
        local_mid = (j - start + end - start) / 2
        x = panel_left_mm + local_mid * cell_width_mm
        if column_meta[j]['study_id'] == 'MANZANAREZ2026':
            figure_text_mm(x, matrix_bottom_mm - 1.4, 'Manzanarez' + chr(10) + '(2026)', ha='center', va='top', fontsize=6.5, linespacing=1.2)
            for k in range(j, end):
                figure_text_mm(panel_left_mm + (k - start + 0.5) * cell_width_mm, matrix_bottom_mm - 8.3, column_meta[k]['timepoint_label'], ha='center', va='top', fontsize=6.5)
        else:
            label = chr(10).join(('(' + line + ')' if line.isdigit() and len(line) == 4 else line for line in column_meta[j]['display_label'].splitlines()))
            figure_text_mm(x, matrix_bottom_mm - 1.4, label, ha='center', va='top', fontsize=6.5, linespacing=1.2)
        j = end
    for i in range(len(genes)):
        for j in range(ncols):
            value = values[i, start + j]
            if np.isnan(value):
                (label, ink) = ('—', '#757575')
            else:
                label = '0.00' if abs(value) < 0.005 else f'{value:.2f}'
                ink = 'white' if abs(value) > 3.4 else '#222222'
            annotation = ax.text(j + 0.5, i + 0.5, label, ha='center', va='center', fontsize=6.2, color=ink)
            annotations.append((ax, i, j, start + j, annotation))
    panels.append((ax, mesh, start, stop))
    panel_left_mm += panel_width_mm + panel_gap_mm
(colorbar_left_mm, colorbar_width_mm) = (39, 83)
footer_center_mm = colorbar_left_mm + colorbar_width_mm / 2
cax = fig.add_axes(rectangle_mm(colorbar_left_mm, 14.5, colorbar_width_mm, 3.5))
cb = fig.colorbar(panels[0][1], cax=cax, orientation='horizontal', ticks=[-6, -3, 0, 3, 6])
cb.set_label('')
cb.outline.set_linewidth(0.35)
cb.ax.tick_params(labelsize=6.5, width=0.5, length=3, pad=2)
figure_text_mm(colorbar_left_mm, 22.1, 'Decrease in stationary', ha='left', va='center', fontsize=6.5, color='#2166ac')
figure_text_mm(colorbar_left_mm + colorbar_width_mm, 22.1, 'Increase in stationary', ha='right', va='center', fontsize=6.5, color='#b2182b')
figure_text_mm(footer_center_mm, 7.2, 'log₂ fold change (stationary / exponential)', ha='center', va='center', fontsize=7)
figure_text_mm(130, 16.25, '—  unavailable / unreliable', ha='left', va='center', fontsize=6.5, color='#444444')
fig.canvas.draw()
renderer = fig.canvas.get_renderer()
for (ax, mesh, start, stop) in panels:
    rendered_colors = mesh.get_facecolors().reshape(len(genes), stop - start, 4)
    for i in range(len(genes)):
        for j in range(stop - start):
            value = values[i, start + j]
            expected_color = matplotlib.colors.to_rgba('#e5e5e5') if np.isnan(value) else cmap(norm(value))
            assert np.allclose(rendered_colors[i, j], expected_color), (i, start + j)
for (ax, i, j, global_col, annotation) in annotations:
    box = annotation.get_window_extent(renderer)
    cell_centre = ax.transData.transform((j + 0.5, i + 0.5))
    assert np.allclose([(box.x0 + box.x1) / 2, (box.y0 + box.y1) / 2], cell_centre, atol=0.1)
for (label, y) in zip(panels[0][0].get_yticklabels(), row_centres):
    box = label.get_window_extent(renderer)
    assert abs((box.y0 + box.y1) / 2 - panels[0][0].transData.transform((0.5, y))[1]) < 0.1
plt.show()
workbook.close()

Morin half-life estimates are author-fitted transcript-decay half-lives, not new decay fits. Positive estimates and author CVs of 0–40 are required at both phases. Schmidt uses the published stationary/exponential abundance ratio. See Measurements and Heatmap columns for endpoint definitions and normalization.